<a href="https://colab.research.google.com/github/LisethTiria/Biosenales-2025-1/blob/main/Proyecto3_ecg/proyecto3_ecg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="display: flex; align-items: center; gap: 20px; background-color: rgba(9, 169, 36, 0.72); padding: 20px; border-radius: 10px; color: white;">
  <img src="logo_udea.png" alt="Logo UdeA" style="height: 80px;">
  <div>
    <h1 style="margin: 0; font-size: 2em;">Proyecto 2 - Filtrado de señales ECG</h1>
    <p style="margin: 2px 0 0 0; font-size: 1.3em;"><strong>Bioseñales y Sistemas</strong></p>
    <p style="margin: 10px 0 0 0; font-size: 1.2em;">Universidad de Antioquia - Facultad de Ingeniería</p>
    <p style="margin: 5px 0 0 0; font-size: 1em;">Integrantes: Luisa Taho, Liseth Tiria, Victor Ocampo</p>
    <p style="margin: 5px 0 0 0; font-size: 1em;">Profesores: John Fredy Ochoa, Juliana Moreno Rada</p>
  </div>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as signal
from scipy import fft
from scipy.fftpack import dct
import matplotlib.pyplot as plt;
import scipy.io as sio;
import seaborn as sns
from scipy.signal import detrend
from scipy.signal.windows import hamming
from scipy.stats import kstest, zscore, kruskal
from scipy.stats import ttest_ind, shapiro, levene, mannwhitneyu
from scipy import stats
import os
import librosa
import neurokit2 as nk
import zipfile
from scipy.signal import welch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Carga el archivo 'Diagnostics.xlsx' desde Google Drive
diag = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Biosenales/Diagnostics.xlsx')

diag

In [ ]:
# Filtrar primero en diag
afib_data = diag[(diag['Rhythm'] == 'AFIB')]
afib_archivos = [f + '.csv' for f in afib_data['FileName'].tolist()]

sb_data = diag[(diag['Rhythm'] == 'SB')]
sb_archivos = [f + '.csv' for f in sb_data['FileName'].tolist()]

sr_data = diag[(diag['Rhythm'] == 'SR')]
sr_archivos = [f + '.csv' for f in sr_data['FileName'].tolist()]

In [ ]:
import os
import zipfile

# Crear carpetas de destino
base_extract_dir = '/content/'
sb_extract_dir = os.path.join(base_extract_dir, 'sb_data')
afib_extract_dir = os.path.join(base_extract_dir, 'afib_data')
sr_extract_dir = os.path.join(base_extract_dir, 'sr_data')
os.makedirs(sb_extract_dir, exist_ok=True)
os.makedirs(afib_extract_dir, exist_ok=True)

# Ruta al archivo ZIP
zip_filepath = '/content/drive/MyDrive/Colab Notebooks/Biosenales/ECGData.zip'

# Mapeo de archivos deseados y sus carpetas
desired_files = {name: sb_extract_dir for name in sb_archivos}
desired_files.update({name: afib_extract_dir for name in afib_archivos})
desired_files.update({name: sr_extract_dir for name in sr_archivos})

# Extraer los archivos desde el ZIP
with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
    for zip_info in zip_ref.infolist():
        filename = os.path.basename(zip_info.filename)
        if filename in desired_files:
            with zip_ref.open(zip_info) as source_file:
                output_path = os.path.join(desired_files[filename], filename)
                with open(output_path, 'wb') as out_file:
                    out_file.write(source_file.read())
            #rint(f"Archivo '{filename}' extraído a '{desired_files[filename]}'")
print(f"Cantidad de archivos para SB (bradicardia sinusal): {len(sb_data)}")
print(f"Cantidad de archivos para AFIB (fibrilación auricular): {len(afib_data)}")
print(f"Cantidad de archivos para SR (ritmo sinusal): {len(sr_data)}")


# Seleccionar 10 señales aleatorias

In [ ]:
import random
import pandas as pd

# Ruta para guardar la lista persistente de señales
selected_files_path = '/content/selected_signals.txt'

# Combinar todas las señales disponibles
all_signals = sb_archivos + afib_archivos + sr_archivos

# Verificar si ya hay un archivo con señales seleccionadas
if os.path.exists(selected_files_path):
    with open(selected_files_path, 'r') as f:
        selected_files = [line.strip() for line in f.readlines()]
else:
    selected_files = random.sample(all_signals, 10)
    with open(selected_files_path, 'w') as f:
        f.writelines(f"{name}\n" for name in selected_files)

# Cargar las señales seleccionadas
selected_signals = []
for file in selected_files:
    if file in sb_archivos:
        path = os.path.join(sb_extract_dir, file)
    elif file in afib_archivos:
        path = os.path.join(afib_extract_dir, file)
    elif file in sr_archivos:
        path = os.path.join(sr_extract_dir, file)
    else:
        continue
    # Leer el archivo CSV (asumiendo que es una columna con la señal)
    signal = pd.read_csv(path, header=None).squeeze()
    selected_signals.append(signal)


# Flujo 1
*   Filtro pasa-altas IIR a 0.5 Hz
*   Filtro wavelet modificado
*   FIltro pasabajas 50 Hz




In [ ]:
import numpy as np
import scipy.signal as signal
import pywt
import matplotlib.pyplot as plt

fs = 500  # frecuencia de muestreo
filtered_signals_f1 = []

for idx, sig in enumerate(selected_signals):
    # 1. Filtro pasaaltas IIR (0.5 Hz)
    b_hp, a_hp = signal.butter(4, 0.5 / (fs / 2), btype='highpass')
    sig_hp = signal.filtfilt(b_hp, a_hp, sig)

    # 2. Filtro wavelet adaptado a ECG
    coeffs = pywt.wavedec(sig_hp, 'db6', level=5)
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    uthresh = sigma * np.sqrt(2 * np.log(len(sig_hp)))
    coeffs_thresh = [pywt.threshold(c, value=uthresh, mode='soft') if i > 0 else c for i, c in enumerate(coeffs)]
    sig_wav = pywt.waverec(coeffs_thresh, 'db6')

    # 3. Filtro pasabajas IIR (50 Hz)
    b_lp, a_lp = signal.butter(4, 50 / (fs / 2), btype='lowpass')
    sig_final = signal.filtfilt(b_lp, a_lp, sig_wav)

    filtered_signals_f1.append(sig_final)


In [ ]:
# Comparación visual
num_signals = len(selected_signals)
plt.figure(figsize=(15, num_signals * 2.5))

for i in range(num_signals):
    plt.subplot(num_signals, 2, 2 * i + 1)
    plt.plot(selected_signals[i], color='gray')
    plt.title(f'Señal Original #{i+1}')
    plt.xlabel('Muestras')
    plt.ylabel('Amplitud')

    plt.subplot(num_signals, 2, 2 * i + 2)
    plt.plot(filtered_signals_f1[i], color='blue')
    plt.title(f'Señal Filtrada - Flujo 1 #{i+1}')
    plt.xlabel('Muestras')

plt.tight_layout()
plt.show()


# Flujo 2
*   Detrend
*   Filtro wavelet modificado
*   FIltro pasabajas IIR 50 Hz

In [ ]:
filtered_signals_f2 = []

for idx, sig in enumerate(selected_signals):
    # 1. Detrend
    sig_detrend = signal.detrend(sig)

    # 2. Filtro wavelet
    coeffs = pywt.wavedec(sig_detrend, 'db6', level=5)
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    uthresh = sigma * np.sqrt(2 * np.log(len(sig)))
    coeffs_thresh = [pywt.threshold(c, value=uthresh, mode='soft') if i > 0 else c for i, c in enumerate(coeffs)]
    sig_wav = pywt.waverec(coeffs_thresh, 'db6')

    # 3. Filtro pasabajas IIR (50 Hz)
    b_lp, a_lp = signal.butter(4, 50 / (fs / 2), btype='low')
    sig_final = signal.filtfilt(b_lp, a_lp, sig_wav)

    filtered_signals_f2.append(sig_final)


# Flujo 3
*   Filtro pasabajas IIR 50 Hz
*   Filtro LOESS
*   Filtro NLM


Dado que los filtros LOESS y NLM no están incluidos en scipy se usará:
*   `statsmodels` para LOESS

*   `skimage.restoration.denoise_nl_means` para NLM

In [ ]:
!pip install statsmodels scikit-image


In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess
from skimage.restoration import denoise_nl_means, estimate_sigma

filtered_signals_f3 = []

for sig in selected_signals:
    # 1. Filtro pasabajas
    b_lp, a_lp = signal.butter(4, 50 / (fs / 2), btype='low')
    sig_lp = signal.filtfilt(b_lp, a_lp, sig)

    # 2. LOESS
    loess_result = lowess(sig_lp, np.arange(len(sig_lp)), frac=0.01, return_sorted=False)

    # 3. NLM
    sigma_est = np.std(loess_result)
    sig_nlm = denoise_nl_means(loess_result, h=1.15 * sigma_est, fast_mode=True, patch_size=5, patch_distance=6, channel_axis=None)

    filtered_signals_f3.append(sig_nlm)


# Comparación visual de los flujos realizados

In [ ]:
def plot_comparisons(originals, f1, f2, f3, title_prefix=''):
    num = len(originals)
    plt.figure(figsize=(18, num * 2.5))

    for i in range(num):
        plt.subplot(num, 4, i * 4 + 1)
        plt.plot(originals[i], color='gray')
        plt.title(f'{title_prefix}Original #{i+1}')
        plt.xlabel('Muestras')

        plt.subplot(num, 4, i * 4 + 2)
        plt.plot(f1[i], color='blue')
        plt.title(f'{title_prefix}Flujo 1 #{i+1}')
        plt.xlabel('Muestras')

        plt.subplot(num, 4, i * 4 + 3)
        plt.plot(f2[i], color='green')
        plt.title(f'{title_prefix}Flujo 2 #{i+1}')
        plt.xlabel('Muestras')

        plt.subplot(num, 4, i * 4 + 4)
        plt.plot(f3[i], color='orange')
        plt.title(f'{title_prefix}Flujo 3 #{i+1}')
        plt.xlabel('Muestras')

    plt.tight_layout()
    plt.show()

# Llamar función para mostrar comparaciones
plot_comparisons(selected_signals, filtered_signals_f1, filtered_signals_f2, filtered_signals_f3)
